![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)

# All imports

In [ ]:
!pip install mujoco

# Set up GPU rendering.
from google.colab import files
import distutils.util
import os
import subprocess
try:
  subprocess.run('nvidia-smi')
  use_GPU = True

except:
  use_GPU = False
  print(
      'Cannot communicate with GPU. ',
      'Make sure you are using a GPU Colab runtime. ',
      'Go to the Runtime menu and select Choose runtime type.')
if use_GPU:
  # Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
  # This is usually installed as part of an Nvidia driver package, but the Colab
  # kernel doesn't install its driver via APT, and as a result the ICD is missing.
  # (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
  NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
  if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
      f.write("""{
      "file_format_version" : "1.0.0",
      "ICD" : {
          "library_path" : "libEGL_nvidia.so.0"
      }
  }
  """)

  # Configure MuJoCo to use the EGL rendering backend (requires GPU)
  print('Setting environment variable to use GPU rendering:')
  %env MUJOCO_GL=egl
else:
  # Required for OSMesa OpenGL driver
  !apt-get update
  !apt-get install -y libosmesa6-dev libgl1-mesa-glx libglfw3

  print('Setting environment variable to use CPU rendering:')
  %env MUJOCO_GL=osmesa

# Check if installation was succesful.
try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

# Other imports and helper functions
import time
import itertools
import numpy as np

# Graphics and plotting.
print('Installing mediapy:')
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

from IPython.display import clear_output
clear_output()


In [ ]:
from skimage.filters import gaussian
from mujoco import viewer

# Simulation

In [ ]:
waterSurface = """
<mujoco>
  <asset>
    <hfield name="terrain" nrow="16" ncol="16" size="5 5 1 1"/>
    <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3"
    rgb2=".2 .3 .4" width="300" height="300" mark="edge" markrgb=".2 .3 .4"/>
    <material name="grid" texture="grid" texrepeat="2 2" texuniform="true"
    reflectance=".2"/>
  </asset>
  <worldbody>
    <light directional="true" diffuse=".9 .9 .9" specular="0.1 0.1 0.1" pos="0 0 5" dir="0 0 -1" castshadow="true"/>
    <geom name="heightfield_geom" type="hfield" hfield="terrain" material="grid" contype="1" conaffinity="1"/>

    <body>
      <freejoint/>
      <geom size="1" pos="0 0 2"/>
    </body>

  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(waterSurface)
data = mujoco.MjData(model)

In [ ]:
duration = 10   # (seconds)
framerate = 30  # (Hz)

height = 480
width = 640
n_frames = int(duration*framerate)
frames = np.zeros((n_frames, height, width, 3), dtype=np.uint8)

mujoco.mj_resetDataKeyframe(model, data, 0)
mujoco.mj_forward(model, data)

with mujoco.Renderer(model, height, width) as renderer:
  for i in range(n_frames):
    while data.time < i/framerate:

      wave = np.ones((16,16), np.float32)
      for j in range(len(wave)):
        wave[:, j] = np.sin(j + data.time)

      wave = (wave + 1 ) / (2)

      model.hfield('terrain').data = wave
      mujoco.mjr_uploadHField(model, renderer._mjr_context, model.hfield('terrain').id)

      mujoco.mj_step(model, data)

    renderer.update_scene(data)
    pixels = renderer.render()
    frames[i] = pixels

media.show_video(frames, fps=framerate)


This browser does not support the video tag.


In [ ]:
waterSurface = """
<mujoco>
  <asset>
    <hfield name="terrain" nrow="16" ncol="16" size="5 5 1 10"/>
    <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3"
    rgb2=".2 .3 .4" width="300" height="300" mark="edge" markrgb=".2 .3 .4"/>
    <material name="grid" texture="grid" texrepeat="2 2" texuniform="true"
    reflectance=".2"/>
  </asset>
  <worldbody>
    <light directional="true" diffuse=".9 .9 .9" specular="0.1 0.1 0.1" pos="0 0 5" dir="0 0 -1" castshadow="true"/>
    <geom name="heightfield_geom" type="hfield" hfield="terrain" material="grid" solref="0 0" solimp="0 0 0 0 0"/>

    <body name="float">
      <freejoint/>
      <geom type="box" size="0.5 0.5 0.5" pos="0 0 2"  solref="0 0" solimp="0 0 0 0 0"/>
    </body>

    <body>
      <freejoint/>
      <geom type="box" size="0.5 0.5 0.5" pos="0 0 5" />
    </body>

  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(waterSurface)
data = mujoco.MjData(model)

In [ ]:
duration = 10   # (seconds)
framerate = 30  # (Hz)

height = 480
width = 640
n_frames = int(duration*framerate)
frames = np.zeros((n_frames, height, width, 3), dtype=np.uint8)

mujoco.mj_resetDataKeyframe(model, data, 0)
mujoco.mj_forward(model, data)


with mujoco.Renderer(model, height, width) as renderer:
  for i in range(n_frames):
    while data.time < i/framerate:

      wave = np.ones((16,16), np.float32)
      for j in range(len(wave)):
        wave[:, j] = np.sin(j + data.time)

      wave = (wave + 1 ) / (2)

      model.hfield('terrain').data = wave
      mujoco.mjr_uploadHField(model, renderer._mjr_context, model.hfield('terrain').id)


      if len(data.contact.dist)> 0:
        mean_dist = np.min(data.contact.dist)
        if mean_dist < 0:
          data.body('float').xfrc_applied[2] = -3000*mean_dist*9.81 - 1000*data.body('float').cvel[5]



      mujoco.mj_step(model, data)

    renderer.update_scene(data)
    pixels = renderer.render()
    frames[i] = pixels

media.show_video(frames, fps=framerate)


This browser does not support the video tag.


In [ ]:
waterSurface = """
<mujoco>
  <asset>
    <hfield name="terrain" nrow="16" ncol="16" size="5 5 1 10"/>
    <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3"
    rgb2=".2 .3 .4" width="300" height="300" mark="edge" markrgb=".2 .3 .4"/>
    <material name="grid" texture="grid" texrepeat="2 2" texuniform="true"
    reflectance=".2" rgba="1 1 1 0.2"/>
  </asset>
  <worldbody>
    <light directional="true" diffuse=".9 .9 .9" specular="0.1 0.1 0.1" pos="0 0 5" dir="0 0 -1" castshadow="true"/>
    <geom name="heightfield_geom" type="hfield" hfield="terrain" material="grid" solref="0 0" solimp="0 0 0 0 0"/>

    <body name="float">
      <freejoint/>
      <geom type="box" size="0.5 0.5 0.5" pos="0 0 2" />
    </body>

  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(waterSurface)
data = mujoco.MjData(model)

In [ ]:
duration = 10   # (seconds)
framerate = 30  # (Hz)

height = 480
width = 640
n_frames = int(duration*framerate)
frames = np.zeros((n_frames, height, width, 3), dtype=np.uint8)

mujoco.mj_resetDataKeyframe(model, data, 0)
mujoco.mj_forward(model, data)


with mujoco.Renderer(model, height, width) as renderer:
  for i in range(n_frames):
    while data.time < i/framerate:

      wave = np.ones((16,16), np.float32)
      for j in range(len(wave)):
        wave[:, j] = np.sin(j + data.time)

      wave = (wave + 1 ) / (2)

      model.hfield('terrain').data = wave
      mujoco.mjr_uploadHField(model, renderer._mjr_context, model.hfield('terrain').id)


      if len(data.contact.dist)> 0:
        mean_dist = np.min(data.contact.dist)
        if mean_dist < 0:
          data.body('float').xfrc_applied[2] = -3000*mean_dist*9.81 - 1000*data.body('float').cvel[5]



      mujoco.mj_step(model, data)

    renderer.update_scene(data)
    pixels = renderer.render()
    frames[i] = pixels

media.show_video(frames, fps=framerate)


This browser does not support the video tag.


In [ ]:
waterSurface = """
<mujoco>
  <asset>
    <hfield name="terrain" nrow="16" ncol="16" size="5 5 1 10"/>
    <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3"
    rgb2=".2 .3 .4" width="300" height="300" mark="edge" markrgb=".2 .3 .4"/>
    <material name="grid" texture="grid" texrepeat="2 2" texuniform="true"
    reflectance=".2" rgba="1 1 1 1"/>
  </asset>
  <worldbody>
    <light directional="true" diffuse=".9 .9 .9" specular="0.1 0.1 0.1" pos="0 0 5" dir="0 0 -1" castshadow="true"/>
    <geom name="heightfield_geom" type="hfield" hfield="terrain" material="grid" conaffinity="0" contype="0" pos="0 0 0"/>

    <body name="float">
      <freejoint/>
      <geom type="box" size="0.5 0.5 0.5" pos="1 -3 2"/>
    </body>
    <body>
      <freejoint/>
      <geom type="box" size="0.5 0.5 0.5" pos="1 -3 5"/>
    </body>

  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(waterSurface)
data = mujoco.MjData(model)

In [ ]:
def getBodyMatrixPosition(hfield, hfiled_geom, body):
  start_x = hfiled_geom.xpos[0] - hfield.size[0]
  start_y = hfiled_geom.xpos[1] - hfield.size[1]

  size_x = 2*hfield.size[0] / hfield.ncol[0]
  size_y = 2*hfield.size[1] / hfield.nrow[0]

  relative_x = body.xipos[0] - start_x
  relative_y = body.xipos[1] - start_y

  col = relative_x // size_x
  row = relative_y // size_y
  return int(row), int(col)


In [ ]:
duration = 10   # (seconds)
framerate = 30  # (Hz)

height = 480
width = 640
n_frames = int(duration*framerate)
frames = np.zeros((n_frames, height, width, 3), dtype=np.uint8)

mujoco.mj_resetDataKeyframe(model, data, 0)
mujoco.mj_forward(model, data)


with mujoco.Renderer(model, height, width) as renderer:
  for i in range(n_frames):
    while data.time < i/framerate:

      wave = np.ones((16,16), np.float32)
      for j in range(len(wave)):
        wave[:, j] = np.sin(j + data.time)

      wave = (wave + 1 ) / (2)

      model.hfield('terrain').data = wave
      mujoco.mjr_uploadHField(model, renderer._mjr_context, model.hfield('terrain').id)


      row, col = getBodyMatrixPosition(model.hfield('terrain'), data.geom('heightfield_geom'), data.body('float'))
      depth = data.body('float').xipos[2] - model.hfield('terrain').data[row, col]
      if depth < 0:
        data.body('float').xfrc_applied[2] = 3000*np.abs(depth)*1*1*9.81 - 1000*data.body('float').cvel[5]



      mujoco.mj_step(model, data)

    renderer.update_scene(data)
    pixels = renderer.render()
    frames[i] = pixels

media.show_video(frames, fps=framerate)

This browser does not support the video tag.
